## Извлечение имен персонажей

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Установка и импорт библиотек

In [ ]:
!pip install natasha

In [ ]:
from pathlib import Path
import pandas as pd #для сохранения в xlsx/csv
import networkx as nx #для сохранения итогового графа
import re
import numpy as np
from scipy.spatial.distance import cosine #для подсчета косинусной близости

from natasha import (
    NamesExtractor,
    PER,
    Doc, #объект для работы с текстов (как nlp в SpaCy)
    NewsEmbedding, #для загрузки предобученной модели
    NewsSyntaxParser, #синтаксический парсер
    NewsMorphTagger, #морфологический парсер (без лемматизации)
    Segmenter, #токенизатор
    NewsNERTagger,
    MorphVocab
)

### Пути к данным

In [ ]:
BASE_PATH = Path('/content/drive/My Drive/Colab Notebooks/Syntaxis_ner') #базовая папка, в которой находится код и данные
DATA_PATH = BASE_PATH / 'data'
file_name = 'Master_Margarita.txt'

### Инициализация объектов из Natasha

In [ ]:
segmenter = Segmenter()
morph_vocab = MorphVocab()

emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
syntax_parser = NewsSyntaxParser(emb)
ner_tagger = NewsNERTagger(emb)

names_extractor = NamesExtractor(morph_vocab)

# <b>1. Извлечение ФИО персонажей</b>

### Вспомогательные функции

Важно понимать, как именно Вы будете разделять на контексты. У Natasha есть особенность: она может

In [ ]:
def open_text(file_path):
    """Открывает текстовый файл"""
    f = open(file_path, 'r', encoding='utf8')
    text = f.read()
    f.close()
    return text


def preprocess_text(text):
    """Предобработка текста"""
    text = re.sub('(?!\n)\s+', ' ', text).strip() #очистка от лишних пробельных символов
    return re.sub('\n+', ' ; \n', text) #

In [ ]:
def get_names_in_text(text):
    """Получает список всех имен из текста"""
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    doc.parse_syntax(syntax_parser)
    doc.tag_ner(ner_tagger)
    for span in doc.spans:
        span.normalize(morph_vocab)
    for span in doc.spans:
        if span.type == PER:
            span.extract_fact(names_extractor)
    #теперь сохраняем результат
    res = []
    for span in doc.spans:
        if span.fact:
            cur_elem = {
                 'normal': span.normal,
                 'start': span.start,
                 'end': span.stop,
                 'first': None,
                 'middle': None,
                 'last': None
            }
            name_info = span.fact.as_dict
            if 'first' in name_info:
                cur_elem['first'] = name_info['first']
            if 'middle' in name_info:
                cur_elem['middle'] = name_info['middle']
            if 'last' in name_info:
                cur_elem['last'] = name_info['last']

            res.append(cur_elem)
    return res


def get_names_in_texts(text):
    """Разбивает текст на контексты и получает списки имен из них"""
    contexts = split_text(text)
    res = []
    for context in contexts:
        res.append(get_names_in_text(context))
    return res

In [ ]:
def save_names_to_csv(names, file_path, sep='|'):
    """Функция для сохранения имен для их дальнейшей проверки"""
    df = pd.DataFrame().from_dict(names)
    df.to_csv(file_path, sep=sep, index=False)

### Чтение текста

In [ ]:
text = open_text(DATA_PATH / file_name)
text = preprocess_text(text)

### Выделение имен

In [ ]:
all_names = get_names_in_text(text)

In [ ]:
print('Кол-во вхождений:', len(all_names))
print('------')
all_names[:10]

Кол-во вхождений: 3655
------


[{'end': 130,
  'first': None,
  'last': 'Гете',
  'middle': None,
  'normal': 'Гете',
  'start': 126},
 {'end': 743,
  'first': 'Михаил',
  'last': 'Берлиоз',
  'middle': 'Александрович',
  'normal': 'Михаил Александрович Берлиоз',
  'start': 715},
 {'end': 952,
  'first': 'Иван',
  'last': 'Понырев',
  'middle': 'Николаевич',
  'normal': 'Иван Николаевич Понырев',
  'start': 929},
 {'end': 987,
  'first': None,
  'last': 'Бездомный',
  'middle': None,
  'normal': 'Бездомный',
  'start': 978},
 {'end': 1542,
  'first': None,
  'last': 'Берлиоз',
  'middle': None,
  'normal': 'Берлиоз',
  'start': 1535},
 {'end': 1669,
  'first': None,
  'last': 'Бездомный',
  'middle': None,
  'normal': 'Бездомный',
  'start': 1660},
 {'end': 1754,
  'first': None,
  'last': 'Берлиоз',
  'middle': None,
  'normal': 'Берлиоз',
  'start': 1747},
 {'end': 1772,
  'first': None,
  'last': 'Абрикосовая',
  'middle': None,
  'normal': 'Абрикосовая',
  'start': 1761},
 {'end': 1860,
  'first': None,
  'last'

Получили список словарей

### Сохранение имен

In [ ]:
save_names_to_csv(all_names, DATA_PATH / 'names.csv', sep='|')

# <b>2. Преобразование списка в такой вид, чтобы все персонажи определялись однозначно (Иван Бездомный = Бездомный)</b>

### Чтение имен

In [ ]:
names = pd.read_csv(DATA_PATH / 'names.csv', sep='|')
names.head()

,normal,start,end,first,middle,last
0,Гете,126,130,NaN,NaN,Гете
1,Михаил Александрович Берлиоз,715,743,Михаил,Александрович,Берлиоз
2,Иван Николаевич Понырев,929,952,Иван,Николаевич,Понырев
3,Бездомный,978,987,NaN,NaN,Бездомный
4,Берлиоз,1535,1542,NaN,NaN,Берлиоз


### Преобразуем к списку словарей (для того, чтобы не путаться)

In [ ]:
names.fillna('?', inplace=True)
all_names = list(names.T.to_dict().values()) #преобразование в список словарей
all_names[:5]

[{'end': 130,
  'first': '?',
  'last': 'Гете',
  'middle': '?',
  'normal': 'Гете',
  'start': 126},
 {'end': 743,
  'first': 'Михаил',
  'last': 'Берлиоз',
  'middle': 'Александрович',
  'normal': 'Михаил Александрович Берлиоз',
  'start': 715},
 {'end': 952,
  'first': 'Иван',
  'last': 'Понырев',
  'middle': 'Николаевич',
  'normal': 'Иван Николаевич Понырев',
  'start': 929},
 {'end': 987,
  'first': '?',
  'last': 'Бездомный',
  'middle': '?',
  'normal': 'Бездомный',
  'start': 978},
 {'end': 1542,
  'first': '?',
  'last': 'Берлиоз',
  'middle': '?',
  'normal': 'Берлиоз',
  'start': 1535}]

Это нетривиальная задача. Если мало данных, то можно сделать вручную с сохраненным списком имен.

Для примера мы будем использовать простую стратегию: из каждой сущности будем извлекать фамилию. Если фамилии нет, то имя. Если имени нет, то отчество.

Если Вы уже решили эту задачу, то этот шаг Вам не нужен

In [ ]:
def extract_uniquely(names):
    """Реализует описанную выше стратегию"""
    res = []
    for name in names:
        #сохраняем позиции в тексте
        cur_name = {
            'start': name['start'],
            'end': name['end']
        }
        if name['last'] != '?':
            cur_name['name'] = name['last']
        elif name['first'] != '?':
            cur_name['name'] = name['first']
        elif name['middle'] != '?':
            cur_name['name'] = name['middle']
        if 'name' in cur_name:
            res.append(cur_name)
    return res

In [ ]:
unique_names = extract_uniquely(all_names)
unique_names[:5]

[{'end': 130, 'name': 'Гете', 'start': 126},
 {'end': 743, 'name': 'Берлиоз', 'start': 715},
 {'end': 952, 'name': 'Понырев', 'start': 929},
 {'end': 987, 'name': 'Бездомный', 'start': 978},
 {'end': 1542, 'name': 'Берлиоз', 'start': 1535}]

Важно: список имен отсортирован по моменту вхождения в текст

## <b>3. Разбиение текста на "контексты" (абзацы/предложения/...) и подсчет количеств совместных употреблений персонажей</b>

При разбиении на контексты удаляется сам разделитель, поэтому его нужно добавить.
Если хотите сделать другое разбиение, то измените реализацию функции split_text

In [ ]:
def split_text(text):
    """"Разбивает текст на контексты"""
    contexts = text.split('\n')
    for i in range(len(contexts)):
        contexts[i] += '\n'
    return contexts

In [ ]:
class GraphMatrix():
    """
    Для удобства работы создадим класс с 2мя полями
    matrix - матрица совместного употребления (столбцы и строки - персонажи, на пересечении - количество совместных соупотреблений)
    name_cnt - словарь <имя, количество употреблений>
    names - список имен
    """
    def __init__(self, matrix, name_cnt, names):
        self.matrix = matrix
        self.name_cnt = name_cnt
        self.names = names

    def update_by_ids(self, new_ids):
        """Обновление данных согласно списку индексов"""
        self.matrix = np.take(self.matrix, new_ids, axis=0)
        self.matrix = np.take(self.matrix, new_ids, axis=1)
        self.names = [self.names[i] for i in new_ids]
        self.name_cnt = {name:self.name_cnt[name] for name in self.names}

    def matrix_to_dict(self):
        res = {name:{} for name in self.names}
        for i in range(self.matrix.shape[0]):
            for j in range(self.matrix.shape[1]):
                if int(self.matrix[i,j]) > 0:
                    res[self.names[i]][self.names[j]] = {'weight': self.matrix[i,j]}
        return res

In [ ]:
def update_matrix_cnt(matrix, name_to_id, context_names):
    """Обновляет матрицу в соответствии с контекстом"""
    for i in range(len(context_names)-1):
        for j in range(i+1, len(context_names)):
            n1 = name_to_id[context_names[i]]
            n2 = name_to_id[context_names[j]]
            matrix[n1][n2] += 1
            matrix[n2][n1] += 1


def build_matrix_cnt(contexts, unique_names):
    """Строит матрицу совместного употребления"""

    #список всех имен
    names = list(set([name['name'] for name in unique_names]))

    #частотный словарь для имен
    name_cnt = {name:0 for name in names}
    for name in unique_names:
        name_cnt[name['name']] += 1

    #совместная встречаемость
    res_matrix = np.zeros((len(names), len(names)))
    name_to_id = {names[i]:i for i in range(len(names))}
    context_ind = 0
    names_ind = 0

    for context in contexts:
        context_names = set()
        cur_names_ind = names_ind
        for i in range(cur_names_ind, len(unique_names)):
            if unique_names[i]['start'] - context_ind < len(context) and\
            unique_names[i]['end'] - context_ind > len(context):
                print(text[unique_names[i]['start']:unique_names[i]['end']])
                print('start, end:', unique_names[i]['start'], unique_names[i]['end'])
                print('name:', unique_names[i]['name'])
                print('context:', context)
                raise Exception('Что-то не так с разбиением. Сущность разделилась')

            if unique_names[i]['start'] - context_ind >= len(context):
                names_ind = i
                break
            names_ind = i + 1
            context_names.add(unique_names[i]['name'])
        update_matrix_cnt(res_matrix, name_to_id, list(context_names))
        if names_ind == len(unique_names):
            break
        context_ind += len(context)
    return GraphMatrix(res_matrix, name_cnt, names)

In [ ]:
contexts = split_text(text)
graph_matrix = build_matrix_cnt(contexts, unique_names)
print('Размер матрицы:', graph_matrix.matrix.shape)

Размер матрицы: (441, 441)


### Фильтрация имен с низкой встречаемостью

In [ ]:
def filter_gapaxes_cnt(graph_matrix, border):
    """Фильтрация имен с низкой встречаемостью в тексте по частоте"""
    new_ids = []
    for i in range(len(graph_matrix.names)):
        if graph_matrix.name_cnt[graph_matrix.names[i]] >= border:
            new_ids.append(i)
    graph_matrix.update_by_ids(new_ids)
    return graph_matrix


def filter_gapaxes_together(graph_matrix, border):
    """Фильтрация имен с низкой совместной встречаемостью"""
    new_ids = []
    for i in range(len(graph_matrix.names)):
        if graph_matrix.matrix[i].sum() >= border:
            new_ids.append(i)
    graph_matrix.update_by_ids(new_ids)
    return graph_matrix

Убираем имена, которые встречались меньше 2 раз (опционально)

In [ ]:
graph_matrix = filter_gapaxes_cnt(graph_matrix, 2)
print('Размер матрицы:', graph_matrix.matrix.shape)

Размер матрицы: (178, 178)


Убираем имена, которые ни разу не употреблялись с другими (опционально)


In [ ]:
graph_matrix = filter_gapaxes_together(graph_matrix, 1)
print('Размер матрицы:', graph_matrix.matrix.shape)

Размер матрицы: (159, 159)


### Построим матрицу на основе схожести в абзацах (вес ребра - косинусная близость)

In [ ]:
def get_weight(vector1, vector2):
    """Косинусная близость между векторами"""
    return (1 - cosine(vector1, vector2)) * 100


def build_by_emb_matrix(emb_matrix):
    """Строит матрицу совместной встречаемости на основе матрицы эмбеддингов"""
    res_matrix = np.zeros((emb_matrix.shape[0], emb_matrix.shape[0]))
    for i in range(emb_matrix.shape[0]-1):
        for j in range(i+1, emb_matrix.shape[0]):
            res_matrix[i,j] = get_weight(emb_matrix[i], emb_matrix[j])
            res_matrix[j,i] = res_matrix[i,j]
    return res_matrix


def build_matrix_similarity(contexts, unique_names):
    """Строит матрицу совместного употребления на основе близости"""
    #список всех имен
    names = list(set([name['name'] for name in unique_names]))

    #частотный словарь для имен
    name_cnt = {name:0 for name in names}
    for name in unique_names:
        name_cnt[name['name']] += 1

    #матрица эмбеддингов
    emb_matrix = np.zeros((len(names), len(contexts)))
    name_to_id = {names[i]:i for i in range(len(names))}
    context_ind = 0
    names_ind = 0

    for context_num in range(len(contexts)):
        context = contexts[context_num]
        cur_names_ind = names_ind
        for i in range(cur_names_ind, len(unique_names)):
            if unique_names[i]['start'] - context_ind < len(context) and\
            unique_names[i]['end'] - context_ind > len(context):
                print(text[unique_names[i]['start']:unique_names[i]['end']])
                print('start, end:', unique_names[i]['start'], unique_names[i]['end'])
                print('name:', unique_names[i]['name'])
                print('context:', context)
                raise Exception('Что-то не так с разбиением. Сущность разделилась')

            if unique_names[i]['start'] - context_ind >= len(context):
                names_ind = i
                break
            names_ind = i + 1
            cur_name_id = name_to_id[unique_names[i]['name']]
            emb_matrix[cur_name_id, context_num] += 1

        if names_ind == len(unique_names):
            break
        context_ind += len(context)

    #совместная встречаемость
    res_matrix = build_by_emb_matrix(emb_matrix)
    return GraphMatrix(res_matrix, name_cnt, names)

In [ ]:
graph_matrix_similarity = build_matrix_similarity(contexts, unique_names)

Фильтрация (опционально)

In [ ]:
graph_matrix_similarity = filter_gapaxes_cnt(graph_matrix_similarity, 2)
graph_matrix_similarity = filter_gapaxes_together(graph_matrix_similarity, 1)
print('Размер матрицы:', graph_matrix_similarity.matrix.shape)

Размер матрицы: (159, 159)


## <b>5. Матрица -> граф</b>

In [ ]:
def save_graph(graph_matrix):
    """Создание и сохранение графа"""
    G = nx.from_dict_of_dicts(graph_matrix.matrix_to_dict()) #создали граф из матрицы
    nx.set_node_attributes(G, graph_matrix.name_cnt, 'weight') #устанавливаем веса вершин
    nx.write_gexf(G, DATA_PATH / 'mm_names_cnt.gexf') #сохраняем в формате для Gephi

In [ ]:
save_graph(graph_matrix)

In [ ]:
save_graph(graph_matrix_similarity)